In [ ]:
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
sns.set_style('whitegrid')
from sklearn.metrics import confusion_matrix , classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Flatten , Conv2D , MaxPooling2D , Dropout , Activation , BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras import regularizers

#Warnings
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy


from tensorflow.keras import layers
from tensorflow.keras.layers import Layer
from tensorflow.keras.regularizers import l2

from tensorflow.keras.constraints import MinMaxNorm
from sklearn.model_selection import train_test_split
from tensorflow.keras.initializers import HeNormal, RandomNormal

In [ ]:
pip install lifelines

In [ ]:
from lifelines.utils import concordance_index

#### Modality 1

In [ ]:
X_train_h = np.load('x_train.npy')
y_train_h = np.load('y_train.npy')
X_test_h = np.load('x_test(1).npy')
y_test_h = np.load('y_test(1).npy')

X_val_h = np.load('x_val.npy')
y_val_h = np.load('y_val.npy')


X_train_h.shape, y_train_h.shape, X_test_h.shape, y_test_h.shape, X_val_h.shape, y_val_h.shape


In [ ]:
random_indices = np.random.choice(1001, 810, replace=False)

X_test_h1 = X_test_h[random_indices]
y_test_h1 = y_test_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape

In [ ]:
random_indices = np.random.choice(1002, 648, replace=False)

X_val_h1 = X_val_h[random_indices]
y_val_h1 = y_val_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape, X_val_h1.shape, y_val_h1.shape

#### Modality 2

In [ ]:
X_train_s = np.load('X.npy')
y_train_s = np.load('Y.npy')

X_train_s.shape, y_train_s.shape

In [ ]:


X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)

X_train_s.shape,X_test_s.shape, y_train_s.shape,y_test_s.shape, y_val_s.shape,y_val_s.shape

In [ ]:


def rotate_image(image, angle):
    """
    Rotate the image by the specified angle.
    """
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_image = cv2.warpAffine(image, rotation_matrix, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return rotated_image

def translate_image(image, tx, ty):
    """
    Translate the image by the specified translation parameters.
    """
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    translated_image = cv2.warpAffine(image, translation_matrix, image.shape[1::-1])
    return translated_image

def apply_gaussian_blur(image, kernel_size=3):
    """
    Apply Gaussian Blur to the image to reduce noise and improve generalization.
    """
    blurred_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return blurred_image

# Augmentation parameters
rotation_angles = [20]
translations = [(5, 5)]
kernel_sizes = [3]  # Gaussian Blur kernel sizes

augmented_X_train = []
augmented_y_train = []

for image, label in zip(X_train_s, y_train_s):
    # Augment with rotations
    for angle in rotation_angles:
        rotated_image = rotate_image(image, angle)
        augmented_X_train.append(rotated_image)
        augmented_y_train.append(label)

    # Augment with translations
    for tx, ty in translations:
        translated_image = translate_image(image, tx, ty)
        augmented_X_train.append(translated_image)
        augmented_y_train.append(label)

    # Augment with Gaussian Blur
    for kernel_size in kernel_sizes:
        blurred_image = apply_gaussian_blur(image, kernel_size)
        augmented_X_train.append(blurred_image)
        augmented_y_train.append(label)

# Convert lists to numpy arrays
augmented_X_train = np.array(augmented_X_train)
augmented_y_train = np.array(augmented_y_train)

# Shuffle the data
shuffle_indices = np.random.permutation(len(augmented_X_train))
augmented_X_train = augmented_X_train[shuffle_indices]
augmented_y_train = augmented_y_train[shuffle_indices]
augmented_X_train.shape, augmented_y_train.shape




In [ ]:
random_indices = np.random.choice(7773, 5421, replace=False)

augmented_X_train = augmented_X_train[random_indices]
augmented_y_train = augmented_y_train[random_indices]

augmented_X_train.shape, augmented_y_train.shape


In [ ]:
X_train_s = np.concatenate((X_train_s, augmented_X_train), axis=0)
y_train_s = np.concatenate((y_train_s, augmented_y_train), axis=0)
X_train_s.shape, y_train_s.shape

In [ ]:
#del augmented_X_train

In [ ]:


# Normalize from [0, 255] to [0, 1]
X_train_s = X_train_s.astype(np.float32) / 255.0
X_test_s = X_test_s.astype(np.float32) / 255.0
X_val_s = X_val_s.astype(np.float32) / 255.0

X_train_s.shape, X_test_s.shape, X_val_s.shape

#### Modality 3

In [ ]:
X_train_multi = np.load('X_train.npy')
X_val_multi = np.load('X_val.npy')
X_test_multi = np.load('X_test.npy')

y_train_multi = np.load('y_train.npy')
y_val_multi = np.load('y_val.npy')
y_test_multi = np.load('y_test.npy')

X_train_multi.shape, X_val_multi.shape, X_test_multi.shape, y_train_multi.shape, y_val_multi.shape, y_test_multi.shape

In [ ]:
y_train_multi = tf.keras.utils.to_categorical(y_train_multi)
y_val_multi = tf.keras.utils.to_categorical(y_val_multi)

y_test_multi = tf.keras.utils.to_categorical(y_test_multi)

y_train_multi.shape, y_val_multi.shape, y_test_multi.shape

##### Modality 4

In [ ]:
X_train_m1 = np.load('X_train_.npy')
y_train_m1 = np.load('y_train_.npy')
X_test_m1 = np.load('X_test_.npy')
y_test_m1 = np.load('y_test_.npy')

X_val_m1 = np.load('X_val_.npy')
y_val_m1 = np.load('y_val_.npy')

X_train_m1.shape, y_train_m1.shape, X_test_m1.shape, y_test_m1.shape, X_val_m1.shape, y_val_m1.shape


In [ ]:
## EXtending MHCF for devsiing SIR module
'''def multi_kernel_groupwise_conv1(x, filters, groups=8, strides=2):
    

    # Multi-Kernel Parallel Convolutions with Different Receptive Fields
    conv1x1 = layers.Conv2D(filters // 4, kernel_size=1, groups=groups, padding="same")(x)
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x)
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, padding="same")(x)
    conv_dilated = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=2, padding="same")(x)

    # Feature Fusion via Concatenation
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5, conv_dilated])

    # Channel Shuffle for Better Feature Mixing
    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)

    # Depthwise + Grouped Convolutions Hybrid
    x1 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x1)
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")(x1)

    # Downsampling x1
    x1 = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x1)

    # **Fix: Ensure x has the same number of channels as x1**
    x = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")(x)
    x = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x)

    # Residual Connection
    x = layers.Add()([x, x1])
    x = layers.Activation('gelu')(x)  # GELU activation for better convergence
    
    return x'''


import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.initializers import Ones, Constant
from tensorflow.keras.metrics import BinaryAccuracy, CategoricalAccuracy, AUC


# -----------------------------------------------------------------------------
# Utility blocks
# -----------------------------------------------------------------------------
class ChannelShuffle(layers.Layer):
    """Efficient channel shuffle with static build‑time shape awareness."""


    def __init__(self, groups: int = 4, **kw):
        super().__init__(**kw)
        self.groups = groups
        
    
    def call(self, x):
        B, H, W, C = tf.unstack(tf.shape(x))
        g = self.groups
        x = tf.reshape(x, [B, H, W, g, C // g])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        return tf.reshape(x, [B, H, W, C])




# -----------------------------------------------------------------------------
# Convolutional primitives
# -----------------------------------------------------------------------------


def stage_groups(filters: int) -> int:
    """Smaller groups for early, wider for late stages."""
    if filters < 128:
        return 4 # more mixing capacity
    if filters < 256:
        return 8
    return 16




def multi_kernel_groupwise_conv1(x, filters, strides=1):
    """Revised MKGC‑1 with single down‑sample point."""
    groups = stage_groups(filters)
    
    
    # Parallel depthwise branches
    conv1x1 = layers.Conv2D(filters // 4, 1, padding="same")(x)
    conv3x3 = layers.DepthwiseConv2D(3, padding="same")(x)
    conv5x5 = layers.DepthwiseConv2D(5, padding="same")(x)
    conv_dil = layers.DepthwiseConv2D(3, dilation_rate=2, padding="same")(x)
    
    
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5, conv_dil])
    x1 = ChannelShuffle(groups)(x1)
    
    
    # Depthwise + PW (no stride yet)
    x1 = layers.DepthwiseConv2D(3, padding="same")(x1)
    x1 = layers.Conv2D(filters, 1, groups=groups, padding="same")(x1)
    
    
    # --- single, shared down‑sampling --------------------------------------
    if strides != 1:
        x = layers.DepthwiseConv2D(3, strides=strides, padding="same")(x)
        x = layers.Conv2D(filters, 1, groups=groups, padding="same")(x)
        x1 = layers.DepthwiseConv2D(3, strides=strides, padding="same")(x1)
        
    
    x = layers.Add()([x, x1])
    return layers.Activation('gelu')(x)



# MHCF for EMRC block
'''def multi_kernel_groupwise_conv3(x, filters, groups=8, strides=2, use_se=True):
    # GPC
    conv1x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, strides=strides, padding="same", use_bias=False)(x)
    conv1x1 = layers.BatchNormalization()(conv1x1)

    # DDC
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=1,strides=strides,  padding="same", use_bias=False)(x)
    
    #conv3x3 = layers.GaussianBlur(kernel_size=3, sigma=1.0)(conv3x3)
    #if strides==2:
     #   conv3x3 = layers.AveragePooling2D(2, 2, padding="same")(conv3x3)

    conv3x3 = layers.BatchNormalization()(conv3x3)

    # DWC
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same", use_bias=False)(x)
    conv5x5 = layers.BatchNormalization()(conv5x5)

    # Concatenation and 1x1 Fusion
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])

    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)

    
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same", use_bias=False)(x1)
    
    x1 = layers.BatchNormalization()(x1)

    # Residual Connection
    x = layers.Conv2D(filters=filters, kernel_size=1, strides=strides, groups= 8, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    x = layers.Add()([x, x1])
    x = layers.Activation('relu')(x)  # GELU activation for better convergence

    return x'''

def multi_kernel_groupwise_conv3(x, filters, strides=1):
    groups = stage_groups(filters)
    # Group‑pointwise conv branch
    gpc = layers.Conv2D(filters, 1, strides=strides, groups=groups, padding="same", use_bias=False)(x)
    gpc = layers.BatchNormalization()(gpc)
    # Depthwise conv branches
    d3 = layers.DepthwiseConv2D(3, strides=strides, padding="same", use_bias=False)(x)
    d3 = layers.BatchNormalization()(d3)
    d5 = layers.DepthwiseConv2D(5, strides=strides, padding="same", use_bias=False)(x)
    d5 = layers.BatchNormalization()(d5)
    
    
    x1 = layers.Concatenate()([gpc, d3, d5])
    x1 = ChannelShuffle(groups)(x1)
    x1 = layers.Conv2D(filters, 1, groups=groups, padding="same", use_bias=False)(x1)
    x1 = layers.BatchNormalization()(x1)
    
    
    # Shortcut with projection if spatial size changes
    if strides != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding="same", use_bias=False)(x)
        shortcut = layers.BatchNormalization()(shortcut)
    else:
        shortcut = x
    
    
    x = layers.Add()([shortcut, x1])
    return layers.Activation('relu')(x)


# Single branch of EMRC for each modality input, such as x_i

'''def RGSA(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x  # Save the original input for residual connection

    # First MHCF
    x = multi_kernel_groupwise_conv3(x, filters=filters, groups=8, strides=strides)

    # Normalization and Activation
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second MHCF
    x = multi_kernel_groupwise_conv3(x, filters=filters, groups=8, strides=(1, 1))  # Strides=1 to avoid mismatch
    x = layers.Conv2D(filters, kernel_size=(1, 1), padding="same")(x)  # Pointwise conv for channel mixing
    x = layers.BatchNormalization()(x)

    # Adjust Shortcut if Needed (Ensure Matching Shape)
    if strides != (1, 1) or use_projection:
        shortcut = layers.Conv2D(filters, kernel_size=(1, 1), strides=strides, groups=8, padding="same")(shortcut)  # Downsampling shortcut
        shortcut = layers.BatchNormalization()(shortcut)

    #x, shortcut = MACFusion(units=filters)([x, shortcut])
    # Residual Connection (Ensure Same Shape)
    x = layers.Add()([x, shortcut])

    # Final Activation
    x = layers.Activation('relu')(x)

    return x
'''



################


import tensorflow as tf
from tensorflow.keras import layers

# -------------------------------------------------------------------
# Custom pooling layers (Min + GlobalMin + Sum + GlobalSum)
# -------------------------------------------------------------------
'''class MinPool2D(layers.Layer):
    """Min pooling via -MaxPool2D(-x)."""
    def __init__(self, pool_size=2, strides=None, padding='valid', data_format=None, **kwargs):
        super().__init__(**kwargs)
        if isinstance(pool_size, int):
            pool_size = (pool_size, pool_size)
        self.pool_size = pool_size
        self.strides = strides if strides is not None else pool_size
        self.padding = padding
        self.data_format = data_format
        self._maxpool = layers.MaxPooling2D(self.pool_size, self.strides, padding=self.padding,
                                            data_format=self.data_format)

    def call(self, x):
        return -self._maxpool(-x)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(pool_size=self.pool_size, strides=self.strides,
                        padding=self.padding, data_format=self.data_format))
        return cfg'''



import tensorflow as tf
from tensorflow.keras import layers

class MinPool2D(layers.Layer):
    """
    Local min pooling over HxW (like MaxPooling2D but for minima).
    Works for NHWC ('channels_last') and NCHW ('channels_first').
    """
    def __init__(self, pool_size=(2, 2), strides=None, padding='SAME',
                 data_format='channels_last', **kwargs):
        super().__init__(**kwargs)
        self.pool_size = tuple(pool_size)
        self.strides   = tuple(strides) if strides is not None else self.pool_size
        self.padding   = padding.upper()
        if self.padding not in ('SAME', 'VALID'):
            raise ValueError("padding must be 'SAME' or 'VALID'")
        if data_format not in ('channels_last', 'channels_first'):
            raise ValueError("data_format must be 'channels_last' or 'channels_first'")
        self.data_format = data_format

    def call(self, inputs):
        x = tf.convert_to_tensor(inputs)
        # Ensure float (negation of ints is problematic for pooling kernels)
        if not x.dtype.is_floating:
            x = tf.cast(x, tf.float32)

        nhwc = (self.data_format == 'channels_last')
        if not nhwc:
            # NCHW -> NHWC for pooling op
            x = tf.transpose(x, [0, 2, 3, 1])

        # min(x) = -max(-x)
        y = -tf.nn.max_pool2d(
            -x,
            ksize=self.pool_size,       # (kh, kw)
            strides=self.strides,       # (sh, sw)
            padding=self.padding        # 'SAME' or 'VALID'
        )

        if not nhwc:
            # NHWC -> NCHW back
            y = tf.transpose(y, [0, 3, 1, 2])
        return y

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "pool_size": self.pool_size,
            "strides": self.strides,
            "padding": self.padding,
            "data_format": self.data_format,
        })
        return cfg


class GlobalMinPooling2D(layers.Layer):
    """Global min pooling across H,W -> shape [B, C]."""
    def __init__(self, keepdims=False, **kwargs):
        super().__init__(**kwargs)
        self.keepdims = bool(keepdims)

    def call(self, x):
        return tf.reduce_min(x, axis=[1, 2], keepdims=self.keepdims)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(keepdims=self.keepdims))
        return cfg


class SumPooling2D(layers.Layer):
    """Sum pooling via AvgPool2D * (kernel_area)."""
    def __init__(self, pool_size=2, strides=None, padding='valid', data_format=None, **kwargs):
        super().__init__(**kwargs)
        if isinstance(pool_size, int):
            pool_size = (pool_size, pool_size)
        self.pool_size = pool_size
        self.strides = strides if strides is not None else pool_size
        self.padding = padding
        self.data_format = data_format
        self._avg = layers.AveragePooling2D(self.pool_size, self.strides, padding=self.padding,
                                            data_format=self.data_format)

    def call(self, x):
        k_h, k_w = self.pool_size
        return self._avg(x) * tf.cast(k_h * k_w, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(pool_size=self.pool_size, strides=self.strides,
                        padding=self.padding, data_format=self.data_format))
        return cfg


class GlobalSumPooling2D(layers.Layer):
    """Global sum across H,W -> shape [B, C]."""
    def __init__(self, keepdims=False, **kwargs):
        super().__init__(**kwargs)
        self.keepdims = bool(keepdims)

    def call(self, x):
        return tf.reduce_sum(x, axis=[1, 2], keepdims=self.keepdims)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(keepdims=self.keepdims))
        return cfg


# -------------------------------------------------------------------
# Converted block: includes SumPooling2D + GlobalSumPooling2D
# Keeps your original variable names and structure
# -------------------------------------------------------------------
def MCSA(x1, filters=None, ratio=16, pool_size=2, strides=2, padding='valid'):   ## Multi-prospective channel statistics attention (MCSA)
    """
    Args:
        x1: Input tensor [B,H,W,C]
        filters: channel count; if None, inferred from x1.shape[-1] (must be known)
        ratio: reduction ratio for SE (filters//ratio)
        pool_size, strides, padding: pooling params for {Min, Avg, Max, Sum}
    """
    if filters is None:
        filters = x1.shape[-1]
    if filters is None:
        raise ValueError("Channel dimension must be known to build Dense layers. "
                         "Pass an Input with explicit channels or provide `filters`.")

    # --- Windowed pooling maps (2x2 by default) ---
    #MN = MinPool2D(pool_size=pool_size, strides=strides, padding=padding)(x1)
    
    MN = MinPool2D()(x1)
    MP = layers.AveragePooling2D(pool_size, strides, padding=padding)(x1)
    AP = layers.MaxPooling2D(pool_size, strides, padding=padding)(x1)
    SP = SumPooling2D(pool_size=pool_size, strides=strides, padding=padding)(x1)  # NEW

    # --- Global descriptors (set 1) ---
    GMN1 = GlobalMinPooling2D()(MN)
    GAP1 = layers.GlobalAveragePooling2D()(MN)
    GMP1 = layers.GlobalMaxPooling2D()(MN)
    GSP1 = GlobalSumPooling2D()(MN)  # NEW
    channel_info1 = GMN1 + GAP1 + GMP1 + GSP1

    
    # --- Global descriptors (set 2) ---
    GMN2 = GlobalMinPooling2D()(MP)
    GAP2 = layers.GlobalAveragePooling2D()(MP)
    GMP2 = layers.GlobalMaxPooling2D()(MP)
    GSP2 = GlobalSumPooling2D()(MP)  # NEW
    channel_info2 = GMN2 + GAP2 + GMP2 + GSP2

    # --- Global descriptors (set 3) ---
    GMN3 = GlobalMinPooling2D()(AP)
    GAP3 = layers.GlobalAveragePooling2D()(AP)
    GMP3 = layers.GlobalMaxPooling2D()(AP)
    GSP3 = GlobalSumPooling2D()(AP)  # NEW
    channel_info3 = GMN3 + GAP3 + GMP3 + GSP3

    # --- Global descriptors (set 3) ---
    GMN4 = GlobalMinPooling2D()(SP)
    GAP4 = layers.GlobalAveragePooling2D()(SP)
    GMP4 = layers.GlobalMaxPooling2D()(SP)
    GSP4 = GlobalSumPooling2D()(SP)  # NEW
    channel_info4 = GMN4 + GAP4 + GMP4 + GSP4

    # Combine three sets (kept as in your snippet)
    div_channel = channel_info1 + channel_info2 + channel_info3 + channel_info4  # shape [B, C]

    # Squeeze-and-Excitation reweighting

    y = layers.Reshape((filters, 1))(div_channel)
    y = layers.Conv1D(
            filters=1, kernel_size=5,
            padding='same', use_bias=False,
            groups=1,           # depth-wise on the channel axis
            activation='sigmoid')(y)

    se = layers.Reshape((1, 1, filters))(y)
    
    #se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(div_channel)
    #se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    #se = layers.Reshape((1, 1, filters))(se)
    
    x1 = layers.Multiply()([x1, se])
    
    return x1



class LowRankDense(layers.Layer):
    def __init__(self, units, rank, use_bias=True, **kw):
        super().__init__(**kw)
        self.units, self.rank, self.use_bias = units, rank, use_bias
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.U = self.add_weight(name="U", shape=(in_dim, self.rank),
                                 initializer="glorot_uniform")
        self.V = self.add_weight(name="V", shape=(self.units, self.rank),
                                 initializer="glorot_uniform")
        if self.use_bias:
            self.b = self.add_weight(name="b", shape=(self.units,),
                                     initializer="zeros")
    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)
        z = tf.linalg.matmul(z, self.V, transpose_b=True)
        if self.use_bias:
            z = z + self.b
        return z


###############




from tensorflow.keras import layers

def RGSA(x,
         filters: int,
         strides=(1, 1),
         *,
         name: str,            # ← required prefix
         groups: int = 16,
         use_projection: bool = False):
    """
    Residual Grouped-Spatial-Attention block with deterministic layer names.

    Args
    ----
    x       : input tensor
    filters : output channels
    strides : spatial down-sampling factor
    name    : **prefix** for every sub-layer, e.g. 'skin_64' or 'lung_256'
    groups  : channel groups in the multi-kernel convs
    """

    shortcut = x  # save input

    # ---------- first multi-kernel conv ----------------------------------
    x = multi_kernel_groupwise_conv3(
            x, filters=filters, strides=strides,
            # give the fusion 1×1 its own name so it can be inspected/frozen
            #name=f"{name}_mkconv1"
        )

    x = MCSA(x, ratio=16, pool_size=2, strides=2, padding='valid')
    
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation('relu', name=f"{name}_relu1")(x)

    # ---------- second conv pair -----------------------------------------
    x = multi_kernel_groupwise_conv3(
            x, filters=filters, strides=(1, 1),
            #name=f"{name}_mkconv2"
        )
    
    x = MCSA(x, ratio=16, pool_size=2, strides=2, padding='valid')
    
    x = layers.Conv2D(filters, 1, padding="same", name=f"{name}_pwconv")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)

    # ---------- projection for the shortcut (if size or #channels change) -
    if strides != (1, 1) or use_projection:
        shortcut = layers.Conv2D(
            filters, 1, strides=strides, padding="same",
            name=f"{name}_proj_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_proj_bn")(shortcut)

    # ---------- residual add & final activation --------------------------
    x = layers.Add(name=f"{name}_add")([x, shortcut])
    x = layers.Activation('relu', name=f"{name}_relu")(x)  # <- **last act**

    return x


In [ ]:
## Old version

import tensorflow as tf
from tensorflow.keras import layers

# ------------------ Utilities ------------------

'''def safe_norm(x, axis=-1, keepdims=False, eps=1e-9):
    return tf.sqrt(tf.maximum(tf.reduce_sum(tf.square(x), axis=axis, keepdims=keepdims), eps))'''

def safe_norm(x, axis=-1, keepdims=False, eps=1e-9):
    return tf.sqrt(tf.maximum(tf.reduce_sum(tf.square(x), axis=axis, keepdims=keepdims), eps))

'''def pairwise_sqdist(x, y):
    x2 = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)
    y2 = tf.reduce_sum(tf.square(y), axis=-1, keepdims=True)
    y2T = tf.transpose(y2, [0, 2, 1])
    d2 = x2 + y2T - 2.0 * tf.matmul(x, y, transpose_b=True)
    return tf.maximum(d2, 0.0)'''

def pairwise_sqdist(x, y):
    # x: [B,N,C], y: [B,M,C] -> [B,N,M]
    x2 = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)      # [B,N,1]
    y2 = tf.reduce_sum(tf.square(y), axis=-1, keepdims=True)      # [B,1,M]
    d2 = x2 + tf.transpose(y2, [0,2,1]) - 2.0 * tf.matmul(x, y, transpose_b=True)
    return tf.maximum(d2, 0.0)

'''def sparsemax(z, axis=-1, eps=1e-12):
    z = tf.convert_to_tensor(z)
    z_sorted = tf.sort(z, axis=axis, direction='DESCENDING')
    z_cumsum = tf.cumsum(z_sorted, axis=axis)
    K = tf.shape(z)[axis]
    k_range = tf.cast(tf.range(1, K+1), z.dtype)
    shape_k = [1]*z.shape.ndims; shape_k[axis] = K
    k = tf.reshape(k_range, shape_k)
    cond = 1 + k * z_sorted > z_cumsum
    k_star = tf.reduce_sum(tf.cast(cond, tf.int32), axis=axis, keepdims=True)
    cumsum_kstar = tf.gather(z_cumsum, k_star-1, axis=axis, batch_dims=z.shape.ndims-1)
    tau = (cumsum_kstar - 1) / tf.cast(k_star, z.dtype)
    p = tf.maximum(z - tau, 0)
    return p / (tf.reduce_sum(p, axis=axis, keepdims=True) + eps)'''

def sparsemax(z, axis=-1, eps=1e-12):
    z = tf.convert_to_tensor(z)
    z_sorted = tf.sort(z, axis=axis, direction='DESCENDING')
    z_cumsum = tf.cumsum(z_sorted, axis=axis)
    K = tf.shape(z)[axis]
    k_range = tf.cast(tf.range(1, K+1), z.dtype)
    shape_k = [1]*z.shape.ndims; shape_k[axis] = K
    k = tf.reshape(k_range, shape_k)
    cond = 1 + k * z_sorted > z_cumsum
    k_star = tf.reduce_sum(tf.cast(cond, tf.int32), axis=axis, keepdims=True)
    cumsum_kstar = tf.gather(z_cumsum, k_star-1, axis=axis, batch_dims=z.shape.ndims-1)
    tau = (cumsum_kstar - 1) / tf.cast(k_star, z.dtype)
    p = tf.maximum(z - tau, 0)
    return p / (tf.reduce_sum(p, axis=axis, keepdims=True) + eps)

# ------------------ Constant-curvature distances ------------------

'''def _map_sphere(x, kappa, eps=1e-6):
    R = tf.math.rsqrt(tf.maximum(kappa, eps))
    n = safe_norm(x, -1, True, eps)
    return R * x / tf.maximum(n, 1e-6), R'''

def _map_sphere(x, kappa, eps=1e-6):
    # kappa: [B,1,1] > 0
    R = tf.math.rsqrt(tf.maximum(kappa, eps))
    n = safe_norm(x, -1, True, eps)
    return R * x / tf.maximum(n, 1e-6), R

'''def _sphere_d2(x, y, R):
    x = x / tf.maximum(safe_norm(x, -1, True), 1e-6) * R
    y = y / tf.maximum(safe_norm(y, -1, True), 1e-6) * R
    cos = tf.clip_by_value(tf.matmul(x, y, transpose_b=True)/(R*R), -1+1e-7, 1-1e-7)
    return tf.square(R * tf.acos(cos))'''

def _sphere_d2(x, y, R):
    x = x / tf.maximum(safe_norm(x, -1, True), 1e-6) * R
    y = y / tf.maximum(safe_norm(y, -1, True), 1e-6) * R
    cos = tf.clip_by_value(tf.matmul(x, y, transpose_b=True)/(R*R),
                           -1+1e-7, 1-1e-7)
    return tf.square(R * tf.acos(cos))


def _map_poincare(x, kappa, eps=1e-6):
    # kappa: [B,1,1] < 0
    R = tf.math.rsqrt(tf.maximum(-kappa, eps))
    n = safe_norm(x, -1, True, eps)
    r = tf.tanh(n/(2*R)) * R
    return r * x / tf.maximum(n, 1e-6), R

'''def _map_poincare(x, kappa, eps=1e-6):
    R = tf.math.rsqrt(tf.maximum(-kappa, eps))
    n = safe_norm(x, -1, True, eps)
    r = tf.tanh(n/(2*R)) * R
    return r * x / tf.maximum(n, 1e-6), R'''

'''def _poincare_d2(u, v, R):
    uu = (tf.reduce_sum(u*u, -1, True)
          + tf.reduce_mean(u*u, -1, True)
          + tf.reduce_max  (u*u, -1, True)
          + tf.reduce_min  (u*u, -1, True))
    vv = (tf.reduce_sum(v*v, -1, True)
          + tf.reduce_mean(v*v, -1, True)
          + tf.reduce_max  (v*v, -1, True)
          + tf.reduce_min  (v*v, -1, True))
    diff2 = pairwise_sqdist(u, v)
    denom = (R*R - uu) * (R*R - tf.transpose(vv, [0,2,1])) + 1e-8
    z = tf.maximum(1 + 2*diff2/denom, 1+1e-6)
    return tf.square(R * tf.acosh(z))'''

def _poincare_d2(u, v, R):
    uu = (tf.reduce_sum(u*u, -1, True)
          + tf.reduce_mean(u*u, -1, True)
          + tf.reduce_max  (u*u, -1, True)
          + tf.reduce_min  (u*u, -1, True))
    vv = (tf.reduce_sum(v*v, -1, True)
          + tf.reduce_mean(v*v, -1, True)
          + tf.reduce_max  (v*v, -1, True)
          + tf.reduce_min  (v*v, -1, True))
    diff2 = pairwise_sqdist(u, v)
    denom = (R*R - uu) * (R*R - tf.transpose(vv, [0,2,1])) + 1e-8
    z = tf.maximum(1 + 2*diff2/denom, 1+1e-6)
    return tf.square(R * tf.acosh(z))

'''def const_curv_pairwise_d2(x, y, kappa):
    kappa = tf.cast(kappa, x.dtype)
    kpos = tf.cast(kappa > 1e-5, x.dtype)
    kneg = tf.cast(kappa < -1e-5, x.dtype)
    kzro = 1 - kpos - kneg
    xs, Rs = _map_sphere(x, tf.maximum(kappa, 1e-5))
    ys, _  = _map_sphere(y, tf.maximum(kappa, 1e-5))
    d2_s   = _sphere_d2(xs, ys, Rs)
    xh, Rh = _map_poincare(x, tf.minimum(kappa, -1e-5))
    yh, _  = _map_poincare(y, tf.minimum(kappa, -1e-5))
    d2_h   = _poincare_d2(xh, yh, Rh)
    d2_e   = pairwise_sqdist(x, y)
    return kpos*d2_s + kneg*d2_h + kzro*d2_e'''

'''def const_curv_pairwise_d2(x, y, kappa):
    kappa = tf.cast(kappa, x.dtype)          # [B]
    kappa = tf.reshape(kappa, [-1,1,1])      # [B,1,1]

    kpos = tf.cast(kappa >  1e-5, x.dtype)    # [B,1,1]
    kneg = tf.cast(kappa < -1e-5, x.dtype)    # [B,1,1]
    kzro = 1 - kpos - kneg                    # [B,1,1]

    # now all branches yield [B,N,M]
    xs, Rs = _map_sphere(x, tf.maximum(kappa, 1e-5))
    ys, _  = _map_sphere(y, tf.maximum(kappa, 1e-5))
    d2_s   = _sphere_d2(xs, ys, Rs)

    xh, Rh = _map_poincare(x, tf.minimum(kappa, -1e-5))
    yh, _  = _map_poincare(y, tf.minimum(kappa, -1e-5))
    d2_h   = _poincare_d2(xh, yh, Rh)

    d2_e   = pairwise_sqdist(x, y)

    # these now broadcast as [B,1,1] * [B,N,M] → [B,N,M]
    return kpos*d2_s + kneg*d2_h + kzro*d2_e'''

def const_curv_pairwise_d2(x, y, kappa, mode="auto"):
    """
    x: [B,N,C], y: [B,M,C], kappa: [B]
    mode: "auto" (shape-safe blend), "euclidean", "spherical", "hyperbolic"
    """
    kappa = tf.cast(kappa, x.dtype)           # [B]
    kappa = tf.reshape(kappa, [-1, 1, 1])     # [B,1,1]

    if mode == "euclidean":
        return pairwise_sqdist(x, y)

    if mode == "spherical":
        kp = tf.maximum(kappa, 1e-5)
        xs, Rs = _map_sphere(x, kp)
        ys, _  = _map_sphere(y, kp)
        return _sphere_d2(xs, ys, Rs)

    if mode == "hyperbolic":
        kn = tf.minimum(kappa, -1e-5)
        xh, Rh = _map_poincare(x, kn)
        yh, _  = _map_poincare(y, kn)
        return _poincare_d2(xh, yh, Rh)

    # ---- "auto": evaluate once, combine with masks (no control-flow) ----
    kp = tf.maximum(kappa, 1e-5)
    kn = tf.minimum(kappa, -1e-5)

    xs, Rs = _map_sphere(x, kp); ys, _  = _map_sphere(y, kp)
    d2_s   = _sphere_d2(xs, ys, Rs)

    xh, Rh = _map_poincare(x, kn); yh, _ = _map_poincare(y, kn)
    d2_h   = _poincare_d2(xh, yh, Rh)

    d2_e   = pairwise_sqdist(x, y)

    kpos = tf.cast(kappa >  1e-5, x.dtype)    # [B,1,1]
    kneg = tf.cast(kappa < -1e-5, x.dtype)    # [B,1,1]
    kzro = 1.0 - kpos - kneg

    return kpos*d2_s + kneg*d2_h + kzro*d2_e



# ------------------ One-pass Sinkhorn ------------------

'''def sinkhorn_one_pass(a, b, C, eps=0.05):
    K = tf.exp(-C/eps)
    P = K * a[..., None]
    P = P / (tf.reduce_sum(P, axis=-1, keepdims=True) + 1e-8)
    P = P * b[:, None, :]
    P = P / (tf.reduce_sum(P, axis=1, keepdims=True) + 1e-8)
    return P'''

def sinkhorn_one_pass(a, b, C, eps=0.05):
    # a: [B,N], b: [B,M], C: [B,N,M]
    K = tf.exp(-C/eps)
    P = K * a[..., None]
    P = P / (tf.reduce_sum(P, axis=-1, keepdims=True) + 1e-8)
    P = P * b[:, None, :]
    P = P / (tf.reduce_sum(P, axis=1, keepdims=True) + 1e-8)
    return P

# ------------------ Building blocks ------------------

'''class SEGate(layers.Layer):
    def __init__(self, d, reduction=8):
        super().__init__()
        self.fc1 = layers.Dense(max(1, d//reduction), activation='relu')
        self.fc2 = layers.Dense(d, activation='sigmoid')
    def call(self, x):
        s = (tf.reduce_mean(x,1,True)
             + tf.reduce_sum(x,1,True)
             + tf.reduce_min(x,1,True)
             + tf.reduce_max(x,1,True))
        s = tf.nn.softmax(s, -1)
        return x * self.fc2(self.fc1(s))'''

class SEGate(layers.Layer):
    def __init__(self, d, reduction=8):
        super().__init__()
        self.fc1 = layers.Dense(max(1, d//reduction), activation='relu')
        self.fc2 = layers.Dense(d, activation='sigmoid')
    def call(self, x):
        # Light but expressive statistics
        s = (tf.reduce_mean(x,1,True)
             + tf.reduce_sum(x,1,True)
             + tf.reduce_min(x,1,True)
             + tf.reduce_max(x,1,True))
        s = tf.nn.softmax(s, -1)
        return x * self.fc2(self.fc1(s))

'''class AtlasCodes(layers.Layer):
    def __init__(self, Nz, d_model):
        super().__init__()
        self.Z = self.add_weight(name = 'atlas', shape=(Nz, d_model),
                                 initializer='random_normal')
        self.P = self.add_weight(name = 'atlas_priors', shape=(Nz, 2),
                                 initializer='random_uniform')
    def call(self, x):
        B = tf.shape(x)[0]
        return tf.tile(self.Z[None], [B,1,1]), tf.tile(self.P[None], [B,1,1])'''

class AtlasCodes(layers.Layer):
    def __init__(self, Nz, d_model):
        super().__init__()
        self.Z = self.add_weight(name='atlas', shape=(Nz, d_model),
                                 initializer='random_normal')
        self.P = self.add_weight(name='atlas_priors', shape=(Nz, 2),
                                 initializer='random_uniform')
    def call(self, x):
        B = tf.shape(x)[0]
        return tf.tile(self.Z[None], [B,1,1]), tf.tile(self.P[None], [B,1,1])
        

'''class DepthwisePointwiseProjector(layers.Layer):
    def __init__(self, H, Dh, kernel_size=3, group=8, **kw):
        super().__init__(**kw)
        self.H, self.Dh = H, Dh
        self.dw = layers.Conv2D(Dh, (kernel_size,1),
                                padding='same',
                                groups=group,
                                use_bias=False)
    def call(self, x):
        B, N, D = tf.unstack(tf.shape(x))
        x4 = tf.reshape(x, [B, N, 1, D])
        y  = self.dw(x4)
        return tf.reshape(y, [B, N, self.H*self.Dh])'''

class DepthwisePointwiseProjector(layers.Layer):
    def __init__(self, H, Dh, kernel_size=3, group=8, **kw):
        super().__init__(**kw)
        self.H, self.Dh = H, Dh
        self.dw = layers.Conv2D(Dh, (kernel_size,1),
                                padding='same',
                                groups=group,
                                use_bias=False)
    def call(self, x):
        # x: [B,N,D] -> [B,N,H*Dh]
        B, N, D = tf.unstack(tf.shape(x))
        x4 = tf.reshape(x, [B, N, 1, D])
        y  = self.dw(x4)
        return tf.reshape(y, [B, N, self.H*self.Dh])



'''class LowRankDense(layers.Layer):
    def __init__(self, units, rank, use_bias=True, **kw):
        super().__init__(**kw)
        self.units, self.rank, self.use_bias = units, rank, use_bias
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.U = self.add_weight(name = "U", shape=(in_dim, self.rank),
                                 initializer="glorot_uniform")
        self.V = self.add_weight(name = "V", shape=(self.units, self.rank),
                                 initializer="glorot_uniform")
        if self.use_bias:
            self.b = self.add_weight(name = "b", shape=(self.units,),
                                     initializer="zeros")
    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)
        z = tf.linalg.matmul(z, self.V, transpose_b=True)
        return z + (self.b if self.use_bias else 0)'''


class LowRankDense(layers.Layer):
    def __init__(self, units, rank, use_bias=True, **kw):
        super().__init__(**kw)
        self.units, self.rank, self.use_bias = units, rank, use_bias
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.U = self.add_weight(name="U", shape=(in_dim, self.rank),
                                 initializer="glorot_uniform")
        self.V = self.add_weight(name="V", shape=(self.units, self.rank),
                                 initializer="glorot_uniform")
        if self.use_bias:
            self.b = self.add_weight(name="b", shape=(self.units,),
                                     initializer="zeros")
    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)
        z = tf.linalg.matmul(z, self.V, transpose_b=True)
        if self.use_bias:
            z = z + self.b
        return z


# ------------------ Hybrid CAPE+ROAM (one-pass) ------------------

class HybridCAPE_ROAM_TF(layers.Layer):
    def __init__(self, d_model, n_heads=1, d_head=None,
                 kappa_max=0.5, alpha=1.0,
                 beta_prior=0.1, gamma_dct=0.1,
                 lam=0.5, topk=8, use_null_sink=True, **kw):
        super().__init__(**kw)
        self.H       = n_heads
        self.Dh      = d_head or d_model//n_heads
        self.d_model = d_model
        self.kappa_max, self.alpha = kappa_max, alpha
        self.lam, self.topk        = lam, topk
        self.use_null_sink         = use_null_sink

        self.WqA    = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WqB    = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WzK    = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WzV    = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        #self.WoA    = layers.Dense(d_model, use_bias=False)
        self.WoA = LowRankDense(d_model, rank=d_model//4, use_bias=False)
        self.WoB    = LowRankDense(d_model, rank=d_model//4, use_bias=False) #layers.Dense(d_model, use_bias=False)
        self.Wa     =  LowRankDense(d_model, rank=d_model//4, use_bias=False) #layers.Dense(d_model, use_bias=False)
        self.Wb     =  LowRankDense(d_model, rank=d_model//4, use_bias=False) #layers.Dense(d_model, use_bias=False)
        self.gateA  = layers.Dense(1, activation='sigmoid')
        self.gateB  = layers.Dense(1, activation='sigmoid')
        self.seA    = SEGate(d_model)
        self.seB    = SEGate(d_model)

        

        
        self.raw_kappa=self.add_weight(name="raw_kappa",shape=(self.H,),
                                       initializer="zeros",trainable=True)
        self.curv_gate  = LowRankDense(3*self.H, rank=(3*self.H)//4, use_bias=False) #layers.Dense(3*self.H, use_bias=False)
        if use_null_sink:
            self.z_null = self.add_weight(name='z_null',
                                          shape=(1, d_model),
                                          initializer='zeros')

    def _split_heads(self, x):
        B,N,_ = tf.unstack(tf.shape(x))
        return tf.transpose(tf.reshape(x, [B,N,self.H,self.Dh]), [0,2,1,3])
    def _merge_heads(self, x):
        x = tf.transpose(x, [0,2,1,3])
        B,N,H,D = tf.unstack(tf.shape(x))
        return tf.reshape(x, [B,N,H*D])

    def call(self, XA, XB, Z, training=False, return_certificates=True):
        B = tf.shape(XA)[0]
        if self.use_null_sink:
            z0 = tf.tile(self.z_null[None], [B,1,1])
            Z  = tf.concat([Z, z0], axis=1)

        QA = self._split_heads(self.WqA(XA))
        QB = self._split_heads(self.WqB(XB))
        ZK = self._split_heads(self.WzK(Z))
        ZV = self._split_heads(self.WzV(Z))

        # geometry scores
        k = tf.reshape(self.kappa_max*tf.tanh(self.raw_kappa), [1,self.H,1,1])
        d2 = const_curv_pairwise_d2(
            tf.reshape(QA, [-1, tf.shape(QA)[2], self.Dh]),
            tf.reshape(ZK, [-1, tf.shape(ZK)[2], self.Dh]),
            tf.repeat(self.raw_kappa, B)
        )
        S = -self.alpha * tf.reshape(d2, [B, self.H, tf.shape(XA)[1], tf.shape(Z)[1]])

        # top-k mask
        '''top_idx = tf.math.top_k(S, k=self.topk).indices
        mask = tf.reduce_max(tf.one_hot(top_idx, depth=tf.shape(S)[-1]), axis=2)
        mask = tf.cast(mask, tf.bool)
        S = tf.where(mask[...,None], S, tf.fill(tf.shape(S), -1e9))'''


        top_idx     = tf.math.top_k(S, k=self.topk).indices    # [B,H,N,topk]
        one_hot_topk= tf.one_hot(
                          top_idx,
                          depth=tf.shape(S)[-1],
                          on_value=True,
                          off_value=False,
                          dtype=tf.bool     # now fully boolean
                      )                         # shape [B,H,N,topk,M]
        mask        = tf.reduce_any(one_hot_topk, axis=3)      # collapse topk → [B,H,N,M]
        S           = tf.where(mask, S, tf.fill(tf.shape(S), -1e9))

        A = tf.nn.softmax(S, axis=-1)

        # CAPE merge
        YA_c = tf.einsum('bhij,bhjd->bhid', A, ZV)
        YA_c = self._merge_heads(YA_c + QA)
        YA_c = self.seA(self.WoA(YA_c))

        w1 = tf.nn.softmax(YA_c)

        YB_c = tf.einsum('bhij,bhjd->bhid', A, ZV)
        YB_c = self._merge_heads(YB_c + QB)
        YB_c = self.seB(self.WoB(YB_c))

        w2 = tf.nn.softmax(YB_c)

        # ROAM one‐pass
        C = pairwise_sqdist(
            tf.math.l2_normalize(self.Wa(XA), axis=-1),
            tf.math.l2_normalize(self.Wb(XB), axis=-1)
        )
        a = tf.nn.softmax(-tf.reduce_sum(self.Wa(XA)**2, -1), axis=-1)
        b = tf.nn.softmax(-tf.reduce_sum(self.Wb(XB)**2, -1), axis=-1)
        P = sinkhorn_one_pass(a, b, C, eps=self.lam)

        mixA = tf.einsum('bmn,bnd->bmd', P, self.Wb(XB))
        mixB = tf.einsum('bmn,bmd->bnd', tf.transpose(P, [0,2,1]), self.Wa(XA))

        YA_r = self.seA(XA + self.lam * mixA)
        w3 = tf.nn.softmax(YA_r)
        
        YB_r = self.seB(XB + self.lam * mixB)
        w4 = tf.nn.softmax(YB_r)

        # final fuse
        #gA = self.gateA(tf.concat([XA, YA_c, YA_r], -1))
        gA = XA + YA_c + YA_r
        w5 = tf.nn.softmax(gA, -1)

        #gB = self.gateB(tf.concat([XB, YB_c, YB_r], -1))
        gB = XB + YB_c + YB_r
        w6 = tf.nn.softmax(gB, -1)
        

        YA = (1 - ((gA * w5)) * YA_c) + ((gA * w6) * YA_r)
        YB = ((1 - (gB * w5)) * YB_c) + ((gB * w6) * YB_r)

        Y1 = (w1 * YA_c) + (w2 * YA_r)
        Y2 = (w1 * YB_c) + (w2 * YB_r)

        Y_1 = YA + YB

        # Per‐channel aggregation *over N only*
        Y_11 = tf.reduce_mean(Y_1, axis=1, keepdims=True)   # → [B,1,C]
        Y_12 = tf.reduce_max(  Y_1, axis=1, keepdims=True)   # → [B,1,C]
        Y_13 = tf.reduce_min(  Y_1, axis=1, keepdims=True)   # → [B,1,C]
        Y_14 = tf.reduce_sum(  Y_1, axis=1, keepdims=True)   # → [B,1,C]
        Y_1  = Y_11 + Y_12 + Y_13 + Y_14                    # → [B,1,C]
        
        # Now do the same for Y_2
        Y_2 = Y1 + Y2   # shape [B, N, C]
        Y_11 = tf.reduce_mean(Y_2, axis=1, keepdims=True)
        Y_12 = tf.reduce_max(  Y_2, axis=1, keepdims=True)
        Y_13 = tf.reduce_min(  Y_2, axis=1, keepdims=True)
        Y_14 = tf.reduce_sum(  Y_2, axis=1, keepdims=True)
        Y_2  = Y_11 + Y_12 + Y_13 + Y_14                    # → [B,1,C] 

        #print('Y_2 shape:', Y_2.shape)
        #print('Y_1 shape:', Y_1.shape)
        
        #w8 = tf.nn.softmax(Y_2, -1)

        attention = tf.nn.sigmoid(Y_1 + Y_2 )

        #print('attention shape:', attention.shape)

        return attention

class HybridSpatialAttention(layers.Layer):
    def __init__(self, d_model, units_q: int = 64, dt_eu=0.5, fractal_dim=1.8,
                 dt_rk=1.0, hidden_mul=2, **kw):
        super().__init__(**kw)
        # drop extra kwargs when instantiating
        self.hybrid = HybridCAPE_ROAM_TF(d_model=d_model)
        self.atlas  = AtlasCodes(Nz=64, d_model=d_model)

        # quantum parameters ψ ∈ ℂ^{D×units_q}
        self.psi_real = self.add_weight(name = "psi_real",
                                        shape=(d_model, units_q),
                                        initializer="glorot_uniform")
        self.psi_imag = self.add_weight(name = "psi_imag",
                                        shape=(d_model, units_q),
                                        initializer="glorot_uniform")
        

    #def build(self):
        #_, _, _, channels = F1
   
        
        self.global_scale1 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
        self.global_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale1')
        #super().build()
        
        self.psi_real = self.add_weight(name="psi_real", shape=(d_model, units_q), initializer="glorot_uniform")
        
        self.psi_imag = self.add_weight(name="psi_imag",
                                            shape=(d_model, units_q),
                                            initializer="glorot_uniform")
    
        # fractal dimension scaling 1 ≤ S ≤ 2
        self.S = self.add_weight(name="fractal_scaling",
                                     shape=(units_q,),
                                     initializer=tf.keras.initializers.Constant(fractal_dim),
                                     constraint=MinMaxNorm(1.0, 2.0))
    
        # simple cross/output gates (channel–wise)
        self.cross_gate  = self.add_weight(name="cross_gate",
                                           shape=(d_model,),
                                           initializer="ones")
        self.output_gate = self.add_weight(name="output_gate",
                                           shape=(d_model,),
                                           initializer="ones")


    # shared tiny MLP for derivative fθ

        '''self.time_head = tf.keras.Sequential([
            layers.GlobalAveragePooling1D(),          # reduce seq dim
            layers.Dense(1, activation="softplus")    #   Δt > 0
        ])

        
        
        self.mlp = tf.keras.Sequential([LowRankDense(hidden_mul * d_model, rank=d_model//4, use_bias=False),
            #layers.Dense(hidden_mul * d_model, activation="gelu"),
            LowRankDense(d_model, rank=d_model//4, use_bias=False) #layers.Dense(d_model)
        ])

        self.dt_eu, self.dt_rk = dt_eu, dt_rk

        # gating to blend Euler vs RK2
        self.blend_gate = layers.Dense(d_model, activation="sigmoid")'''

    # -------------------------------------------------------------------
    '''def _adaptive_dt(self, h):
        """
        Computes Δt for each batch item and reshapes to [B,1,1].
        Works for both [B,N,C] or [B,H,W,C] tensors (channel last).
        """
        if len(h.shape) == 3:          # [B,N,C]
            mean_t = self.time_head(h)           # GAP over N
        else:                          # [B,H,W,C]
            B, H, W, C = tf.unstack(tf.shape(h))
            mean_t = self.time_head(tf.reshape(h, [B, H*W, C]))
        return tf.reshape(mean_t, [-1, 1, 1])    # broadcastable'''


    def _adaptive_dt(self, h):
        # scalar per-sample step size
        if len(h.shape) == 3:                     # [B,N,C]
            step = self.time_head(h)              # [B,1]
        else:                                     # [B,H,W,C]
            B,H,W,C = tf.unstack(tf.shape(h))
            step = self.time_head(tf.reshape(h, [B, H*W, C]))  # [B,1]
    
        # add (rank-1) singleton axes so shape = [B,1,1,…,1]
        shape = [-1] + [1] * (len(h.shape) - 1)
        return tf.reshape(step, shape)

    # ------------------------------------------------------------------ #
    def euler(self, h):
        Δt = self._adaptive_dt(h)      # [B,1,1]
        dh = self.mlp(h)               # same shape as h
        return h + Δt * dh

    def rk2(self, h):
        Δt = self._adaptive_dt(h)
        k1  = self.mlp(h)
        mid = h + (Δt / 2.0) * k1
        k2  = self.mlp(mid)
        return h + Δt * k2

    

    '''def _euler(self, h):
        return h + self.dt_eu * self.mlp(h)

    def _rk2(self, h):
        k1  = self.mlp(h)
        mid = h + (self.dt_rk / 2.0) * k1
        k2  = self.mlp(mid)
        return h + self.dt_rk * k2'''

    
    # quantum parameters ψ ∈ ℂ^{D×units_q}
    '''self.psi_real = self.add_weight("psi_real",
                                        shape=(d_model, units_q),
                                        initializer="glorot_uniform")'''
    
    # --------------------------------------------------------------------- #
    def _expand(self, v, target):
        """Broadcast [B,C] → [B,H,W,C] to match feature map."""
        if len(target.shape) == 4:
            v = v[:, None, None, :]                # [B,1,1,C]
            return tf.tile(v, [1, target.shape[1], target.shape[2], 1])
        return v                                   # already flat
    
    
    

    def call(self, F1, F2, training=False):
        def wm(F):
            avg = tf.reduce_mean(F, [1,2], keepdims=True)
            mx  = tf.reduce_max(F, [1,2], keepdims=True)
            mn  = tf.reduce_min(F, [1,2], keepdims=True)
            sm  = tf.reduce_sum(F, [1,2], keepdims=True)
            return tf.nn.softmax(avg+mx+mn+sm, -1)*F

        
        # 3. Euler and RK2 refinements *in parallel*
        '''x1_eu, x2_eu = self.euler(F1), self.euler(F2)
        x1_rk, x2_rk = self.rk2(F1),  self.rk2(F2)

        # 4. Learnable blend
        #    gate g ∈ (0,1): output = g·Euler + (1-g)·RK2
        g1 = self.blend_gate(tf.reduce_mean(F1,[1,2]))[:,None,None,:]  # [B,1,1,C]
        g2 = self.blend_gate(tf.reduce_mean(F2,[1,2]))[:,None,None,:]

        F11 = g1 * x1_eu + (1.0 - g1) * x1_rk
        F22 = g2 * x2_eu + (1.0 - g2) * x2_rk'''
        

        
        F1w = wm(F1)
        F2w = wm(F2)
        B,H,W,C = tf.unstack(tf.shape(F1w))
        B2,H2,W2,C2 = tf.unstack(tf.shape(F2w))
        XA = tf.reshape(F1w, [B, H*W, C])
        XB = tf.reshape(F2w, [B2, H2*W2, C2])
        Z, _ = self.atlas(F1w)

        attn_core = self.hybrid(XA, XB, Z, training=training, return_certificates=False)
        
        attn_core = attn_core[:, None, :, :]  # → [B,1,1,C]
        #print('attention now shape:', attention.shape)
        #attention = tf.reshape(attention, [B, H, W, C])




        ###############

        x1_flat = tf.signal.dct(tf.reduce_mean(F1, [1,2]), type=2, norm='ortho')  # [B,C]
        x2_flat = tf.signal.dct(tf.reduce_mean(F2, [1,2]), type=2, norm='ortho')

        ψ1 = tf.complex(tf.matmul(x1_flat, self.psi_real),
                        tf.matmul(x1_flat, self.psi_imag))
        ψ2 = tf.complex(tf.matmul(x2_flat, self.psi_real),
                        tf.matmul(x2_flat, self.psi_imag))

        p1 = tf.math.real(ψ1 * tf.math.conj(ψ1))             # |ψ|²
        p2 = tf.math.real(ψ2 * tf.math.conj(ψ2))

        # 3. Fractal attention (power-law weighting) ----------------------
        #    scaled_p = p^(S)  (element-wise; S ∈ [1,2])
        scaled1 = tf.pow(p1 + 1e-8, self.S)
        scaled2 = tf.pow(p2 + 1e-8, self.S)

        att_q1 = tf.nn.softmax(scaled1, axis=-1)             # [B,units_q]
        att_q2 = tf.nn.softmax(scaled2, axis=-1)

        # Project attentions back to channel space via ψ_real -------------
        v1 = tf.matmul(att_q1, tf.transpose(self.psi_real))  # [B,C]
        v2 = tf.matmul(att_q2, tf.transpose(self.psi_real))  # [B,C]

        v1 = self._expand(v1, F1)                            # [B,H,W,C]
        v2 = self._expand(v2, F2)

        # 4. Cross-fusion & channel gating -------------------------------
        cg   = tf.reshape(self.cross_gate,  (1,1,1,-1))
        gout = tf.reshape(self.output_gate, (1,1,1,-1))

        out1 = gout * (F2 * cg * v1)     # F2 guiding F1
        out2 = gout * (F1 * cg * v2)     # F1 guiding F2

        # 5. Fuse with CAPE-ROAM mask and return --------------------------
        return out1 * attn_core + F1, out2 * attn_core + F2

        ##############

        '''x1 = F1 * attention # * self.global_scale1
        x2 = F2 * attention #* self.global_scale2

        #print('attention x1 shape:', x1.shape)
        #print('attention x2 shape:', x2.shape)
        return x1, x2 #tf.reshape(YA, [B,H,W,C]), tf.reshape(YB, [B,H,W,C])'''


In [ ]:
'''def EMRC(x1, x2, filters, strides):
    #x1 = RGSA(x1, filters=filters, strides=strides, use_projection=True)

    x1 = RGSA(x1, filters=filters, strides=strides, name='m1_f"{filters}_pool', use_projection=True)

    x1 = RGSA(x1, filters=filters, strides=strides, name='m2_f"{filters}_pool', use_projection=True)
    
    #x2 = RGSA(x2, filters=filters, strides=strides, use_projection=True)
    return x1, x2'''

In [ ]:
'''def EMRC(x1, x2, filters, strides):
    
    # ── Stage-1 refinement ────────────────────────────────────────────────
    x1 = RGSA(
        x1,
        filters=filters,
        strides=strides,
        name=f"m1_f{filters}_pool",
        use_projection=True,
    )
    x2 = RGSA(
        x2,
        filters=filters,
        strides=strides,
        name=f"m2_f{filters}_pool",
        use_projection=True,
    )

    return x1, x2
'''

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.constraints import MinMaxNorm

# ------------------ Utilities ------------------

def safe_norm(x, axis=-1, keepdims=False, eps=1e-9):
    return tf.sqrt(tf.maximum(tf.reduce_sum(tf.square(x), axis=axis, keepdims=keepdims), eps))

def pairwise_sqdist(x, y):
    # x: [B,N,C], y: [B,M,C] -> [B,N,M]
    x2 = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)      # [B,N,1]
    y2 = tf.reduce_sum(tf.square(y), axis=-1, keepdims=True)      # [B,1,M]
    d2 = x2 + tf.transpose(y2, [0,2,1]) - 2.0 * tf.matmul(x, y, transpose_b=True)
    return tf.maximum(d2, 0.0)

def sparsemax(z, axis=-1, eps=1e-12):
    z = tf.convert_to_tensor(z)
    z_sorted = tf.sort(z, axis=axis, direction='DESCENDING')
    z_cumsum = tf.cumsum(z_sorted, axis=axis)
    K = tf.shape(z)[axis]
    k_range = tf.cast(tf.range(1, K+1), z.dtype)
    shape_k = [1]*z.shape.ndims; shape_k[axis] = K
    k = tf.reshape(k_range, shape_k)
    cond = 1 + k * z_sorted > z_cumsum
    k_star = tf.reduce_sum(tf.cast(cond, tf.int32), axis=axis, keepdims=True)
    cumsum_kstar = tf.gather(z_cumsum, k_star-1, axis=axis, batch_dims=z.shape.ndims-1)
    tau = (cumsum_kstar - 1) / tf.cast(k_star, z.dtype)
    p = tf.maximum(z - tau, 0)
    return p / (tf.reduce_sum(p, axis=axis, keepdims=True) + eps)

# ------------------ Constant-curvature distances ------------------

def _map_sphere(x, kappa, eps=1e-6):
    # kappa: [B,1,1] > 0
    R = tf.math.rsqrt(tf.maximum(kappa, eps))
    n = safe_norm(x, -1, True, eps)
    return R * x / tf.maximum(n, 1e-6), R

def _sphere_d2(x, y, R):
    x = x / tf.maximum(safe_norm(x, -1, True), 1e-6) * R
    y = y / tf.maximum(safe_norm(y, -1, True), 1e-6) * R
    cos = tf.clip_by_value(tf.matmul(x, y, transpose_b=True)/(R*R),
                           -1+1e-7, 1-1e-7)
    return tf.square(R * tf.acos(cos))

def _map_poincare(x, kappa, eps=1e-6):
    # kappa: [B,1,1] < 0
    R = tf.math.rsqrt(tf.maximum(-kappa, eps))
    n = safe_norm(x, -1, True, eps)
    r = tf.tanh(n/(2*R)) * R
    return r * x / tf.maximum(n, 1e-6), R

def _poincare_d2(u, v, R):
    uu = (tf.reduce_sum(u*u, -1, True)
          + tf.reduce_mean(u*u, -1, True)
          + tf.reduce_max  (u*u, -1, True)
          + tf.reduce_min  (u*u, -1, True))
    vv = (tf.reduce_sum(v*v, -1, True)
          + tf.reduce_mean(v*v, -1, True)
          + tf.reduce_max  (v*v, -1, True)
          + tf.reduce_min  (v*v, -1, True))
    diff2 = pairwise_sqdist(u, v)
    denom = (R*R - uu) * (R*R - tf.transpose(vv, [0,2,1])) + 1e-8
    z = tf.maximum(1 + 2*diff2/denom, 1+1e-6)
    return tf.square(R * tf.acosh(z))

def const_curv_pairwise_d2(x, y, kappa, mode="auto"):
    """
    x: [B,N,C], y: [B,M,C], kappa: [B]
    mode: "auto" (shape-safe blend), "euclidean", "spherical", "hyperbolic"
    """
    kappa = tf.cast(kappa, x.dtype)           # [B]
    kappa = tf.reshape(kappa, [-1, 1, 1])     # [B,1,1]

    if mode == "euclidean":
        return pairwise_sqdist(x, y)

    if mode == "spherical":
        kp = tf.maximum(kappa, 1e-5)
        xs, Rs = _map_sphere(x, kp)
        ys, _  = _map_sphere(y, kp)
        return _sphere_d2(xs, ys, Rs)

    if mode == "hyperbolic":
        kn = tf.minimum(kappa, -1e-5)
        xh, Rh = _map_poincare(x, kn)
        yh, _  = _map_poincare(y, kn)
        return _poincare_d2(xh, yh, Rh)

    # ---- "auto": evaluate once, combine with masks (no control-flow) ----
    kp = tf.maximum(kappa, 1e-5)
    kn = tf.minimum(kappa, -1e-5)

    xs, Rs = _map_sphere(x, kp); ys, _  = _map_sphere(y, kp)
    d2_s   = _sphere_d2(xs, ys, Rs)

    xh, Rh = _map_poincare(x, kn); yh, _ = _map_poincare(y, kn)
    d2_h   = _poincare_d2(xh, yh, Rh)

    d2_e   = pairwise_sqdist(x, y)

    kpos = tf.cast(kappa >  1e-5, x.dtype)    # [B,1,1]
    kneg = tf.cast(kappa < -1e-5, x.dtype)    # [B,1,1]
    kzro = 1.0 - kpos - kneg

    return kpos*d2_s + kneg*d2_h + kzro*d2_e



# ------------------ One-pass Sinkhorn ------------------

def sinkhorn_one_pass(a, b, C, eps=0.05):
    # a: [B,N], b: [B,M], C: [B,N,M]
    K = tf.exp(-C/eps)
    P = K * a[..., None]
    P = P / (tf.reduce_sum(P, axis=-1, keepdims=True) + 1e-8)
    P = P * b[:, None, :]
    P = P / (tf.reduce_sum(P, axis=1, keepdims=True) + 1e-8)
    return P

# ------------------ Building blocks ------------------

class SEGate(layers.Layer):
    def __init__(self, d, reduction=8):
        super().__init__()
        self.fc1 = layers.Dense(max(1, d//reduction), activation='relu')
        self.fc2 = layers.Dense(d, activation='sigmoid')
    def call(self, x):
        # Light but expressive statistics
        s = (tf.reduce_mean(x,1,True)
             + tf.reduce_sum(x,1,True)
             + tf.reduce_min(x,1,True)
             + tf.reduce_max(x,1,True))
        s = tf.nn.softmax(s, -1)
        return x * self.fc2(self.fc1(s))

class AtlasCodes(layers.Layer):
    def __init__(self, Nz, d_model):
        super().__init__()
        self.Z = self.add_weight(name='atlas', shape=(Nz, d_model),
                                 initializer='random_normal')
        self.P = self.add_weight(name='atlas_priors', shape=(Nz, 2),
                                 initializer='random_uniform')
    def call(self, x):
        B = tf.shape(x)[0]
        return tf.tile(self.Z[None], [B,1,1]), tf.tile(self.P[None], [B,1,1])

class DepthwisePointwiseProjector(layers.Layer):
    def __init__(self, H, Dh, kernel_size=3, group=8, **kw):
        super().__init__(**kw)
        self.H, self.Dh = H, Dh
        self.dw = layers.Conv2D(Dh, (kernel_size,1),
                                padding='same',
                                groups=group,
                                use_bias=False)
    def call(self, x):
        # x: [B,N,D] -> [B,N,H*Dh]
        B, N, D = tf.unstack(tf.shape(x))
        x4 = tf.reshape(x, [B, N, 1, D])
        y  = self.dw(x4)
        return tf.reshape(y, [B, N, self.H*self.Dh])

class LowRankDense(layers.Layer):
    def __init__(self, units, rank, use_bias=True, **kw):
        super().__init__(**kw)
        self.units, self.rank, self.use_bias = units, rank, use_bias
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.U = self.add_weight(name="U", shape=(in_dim, self.rank),
                                 initializer="glorot_uniform")
        self.V = self.add_weight(name="V", shape=(self.units, self.rank),
                                 initializer="glorot_uniform")
        if self.use_bias:
            self.b = self.add_weight(name="b", shape=(self.units,),
                                     initializer="zeros")
    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)
        z = tf.linalg.matmul(z, self.V, transpose_b=True)
        if self.use_bias:
            z = z + self.b
        return z

# ------------------ Hybrid CAPE+ROAM (optimized) ------------------

class HybridCAPE_ROAM_TF(layers.Layer):
    def __init__(self, d_model, n_heads=1, d_head=None,
                 kappa_max=0.5, alpha=1.0,
                 beta_prior=0.1, gamma_dct=0.1,
                 lam=0.0,               # <-- CHANGED: default off to avoid N×N
                 topk=8, use_null_sink=True, geometry="auto",
                 **kw):
        super().__init__(**kw)
        self.H       = n_heads
        self.Dh      = d_head or d_model//n_heads
        self.d_model = d_model
        self.kappa_max, self.alpha = kappa_max, alpha
        self.lam      = float(lam)     # gate ROAM work at runtime
        self.topk     = int(topk)
        self.use_null_sink = use_null_sink
        self.geometry = geometry

        self.WqA = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WqB = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WzK = DepthwisePointwiseProjector(self.H, self.Dh, 1)
        self.WzV = DepthwisePointwiseProjector(self.H, self.Dh, 1)

        self.WoA = LowRankDense(d_model, rank=max(1, d_model//4), use_bias=False)
        self.WoB = LowRankDense(d_model, rank=max(1, d_model//4), use_bias=False)
        self.Wa  = LowRankDense(d_model, rank=max(1, d_model//4), use_bias=False)
        self.Wb  = LowRankDense(d_model, rank=max(1, d_model//4), use_bias=False)

        self.gateA = layers.Dense(1, activation='sigmoid')
        self.gateB = layers.Dense(1, activation='sigmoid')
        self.seA   = SEGate(d_model)
        self.seB   = SEGate(d_model)

        self.raw_kappa = self.add_weight(name="raw_kappa",
                                         shape=(self.H,),
                                         initializer="zeros", trainable=True)
        self.curv_gate = LowRankDense(3*self.H, rank=max(1, (3*self.H)//4),
                                      use_bias=False)

        self.mix_logit = self.add_weight(name = "mix_logit", shape=(), initializer="zeros")
        
        if use_null_sink:
            self.z_null = self.add_weight(name='z_null',
                                          shape=(1, d_model),
                                          initializer='zeros')

    def _split_heads(self, x):
        B,N,_ = tf.unstack(tf.shape(x))
        return tf.transpose(tf.reshape(x, [B,N,self.H,self.Dh]), [0,2,1,3])
    def _merge_heads(self, x):
        x = tf.transpose(x, [0,2,1,3])
        B,N,H,D = tf.unstack(tf.shape(x))
        return tf.reshape(x, [B,N,H*D])

    def call(self, XA, XB, Z, training=False, return_certificates=True):
        B = tf.shape(XA)[0]
        if self.use_null_sink:
            z0 = tf.tile(self.z_null[None], [B,1,1])
            Z  = tf.concat([Z, z0], axis=1)  # [B, M+1, C]

        QA = self._split_heads(self.WqA(XA))    # [B,H,N,Dh]
        QB = self._split_heads(self.WqB(XB))    # [B,H,N,Dh]
        ZK = self._split_heads(self.WzK(Z))     # [B,H,M,Dh]
        ZV = self._split_heads(self.WzV(Z))     # [B,H,M,Dh]

        # ---------- CAPE scores (geometry-aware) ----------
        k = tf.reshape(self.kappa_max*tf.tanh(self.raw_kappa), [1,self.H,1,1])  # [1,H,1,1]

        g = tf.sigmoid(self.mix_logit)   # learnable in (0,1)
        QC = g * QA + (1.0 - g) * QB

        '''d2 = const_curv_pairwise_d2(
            tf.reshape(QA, [-1, tf.shape(QA)[2], self.Dh]),  # [B*H, N, Dh]
            tf.reshape(ZK, [-1, tf.shape(ZK)[2], self.Dh]),  # [B*H, M, Dh]
            tf.repeat(self.raw_kappa, B)                     # [B*H]
        )'''

        '''d2 = const_curv_pairwise_d2(
            tf.reshape(QC, [-1, tf.shape(QC)[2], self.Dh]),   # [B*H, N, Dh]
            tf.reshape(ZK, [-1, tf.shape(ZK)[2], self.Dh]),   # [B*H, M, Dh]
            tf.repeat(self.raw_kappa, B)                      # [B*H]
        )'''

        d2 = const_curv_pairwise_d2(
        tf.reshape(QC, [-1, tf.shape(QC)[2], self.Dh]),
        tf.reshape(ZK, [-1, tf.shape(ZK)[2], self.Dh]),
        tf.repeat(self.raw_kappa, tf.shape(QC)[0] // tf.shape(QA)[0]),
        mode=self.geometry
        )

        S = -self.alpha * tf.reshape(d2, [B, self.H, tf.shape(XA)[1], tf.shape(Z)[1]])
        
        #S = -self.alpha * tf.reshape(d2, [B, self.H, tf.shape(XA)[1], tf.shape(Z)[1]])  # [B,H,N,M]

        #S = -self.alpha * tf.reshape(d2, [B, self.H, tf.shape(XA)[1], tf.shape(Z)[1]])

        # ---------- top-k masking ----------
        #M = tf.shape(S)[-1]
        #k_eff = tf.minimum(tf.cast(self.topk, tf.int32), M)

        M = tf.shape(S)[-1]
        k_eff = tf.maximum(1, tf.minimum(tf.cast(self.topk, tf.int32), M))  # k >= 1

        '''def mask_topk(S_):
            idx = tf.math.top_k(S_, k=k_eff).indices          # [B,H,N,k]
            oh  = tf.one_hot(idx, depth=M, dtype=tf.bool)     # [B,H,N,k,M]
            mask= tf.reduce_any(oh, axis=3)                   # [B,H,N,M]
            return tf.where(mask, S_, tf.fill(tf.shape(S_), tf.constant(-1e9, S_.dtype)))'''

        idx = tf.math.top_k(S, k=k_eff).indices                 # [B,H,N,k]
        oh  = tf.one_hot(idx, depth=M, on_value=True, off_value=False, dtype=tf.bool)
        mask= tf.reduce_any(oh, axis=3)                         # [B,H,N,M]
        
        big_neg = tf.cast(-1e9, S.dtype)
        S = tf.where(mask, S, tf.fill(tf.shape(S), big_neg))

        A = tf.nn.softmax(S, axis=-1)                         # [B,H,N,M]

        # ---------- CAPE merge ----------
        YA_c = tf.einsum('bhij,bhjd->bhid', A, ZV)            # [B,H,N,Dh]
        YA_c = self._merge_heads(YA_c + QA)                   # [B,N,C]
        YA_c = self.seA(self.WoA(YA_c))                       # [B,N,C]

        YB_c = tf.einsum('bhij,bhjd->bhid', A, ZV)
        YB_c = self._merge_heads(YB_c + QB)
        YB_c = self.seB(self.WoB(YB_c))                       # [B,N,C]

        # ---------- ROAM (gated; heavy path) ----------
        # Compute Wa/Wb ONCE and reuse
        YA_r = self.seA(XA)  # default if lam=0
        YB_r = self.seB(XB)

        if self.lam > 1e-6:
            Wa_XA = self.Wa(XA)                               # [B,N,C]
            Wb_XB = self.Wb(XB)                               # [B,N,C]
            Wa_XA_n = tf.math.l2_normalize(Wa_XA, axis=-1)
            Wb_XB_n = tf.math.l2_normalize(Wb_XB, axis=-1)

            C = pairwise_sqdist(Wa_XA_n, Wb_XB_n)             # [B,N,N]
            a = tf.nn.softmax(-tf.reduce_sum(Wa_XA**2, -1), axis=-1)  # [B,N]
            b = tf.nn.softmax(-tf.reduce_sum(Wb_XB**2, -1), axis=-1)  # [B,N]
            P = sinkhorn_one_pass(a, b, C, eps=float(self.lam))       # [B,N,N]

            mixA = tf.einsum('bmn,bnd->bmd', P, Wb_XB)
            mixB = tf.einsum('bmn,bmd->bnd', tf.transpose(P, [0,2,1]), Wa_XA)

            YA_r = self.seA(XA + self.lam * mixA)
            YB_r = self.seB(XB + self.lam * mixB)

        # ---------- final fuse ----------
        w1 = tf.nn.softmax(YA_c)   # [B,N,C]
        w2 = tf.nn.softmax(YB_c)
        gA = XA + YA_c + YA_r
        gB = XB + YB_c + YB_r
        w5 = tf.nn.softmax(gA, axis=-1)
        w6 = tf.nn.softmax(gB, axis=-1)

        YA = (1 - ((gA * w5)) * YA_c) + ((gA * w6) * YA_r)
        YB = ((1 - (gB * w5)) * YB_c) + ((gB * w6) * YB_r)

        Y1 = (w1 * YA_c) + (w2 * YA_r)
        Y2 = (w1 * YB_c) + (w2 * YB_r)
        Y_1 = YA + YB                    # [B,N,C]

        # channel summaries (over N)
        def redN(t):
            return (tf.reduce_mean(t, 1, True) +
                    tf.reduce_max (t, 1, True) +
                    tf.reduce_min (t, 1, True) +
                    tf.reduce_sum (t, 1, True))  # [B,1,C]
        Y_1 = redN(Y_1)
        Y_2 = redN(Y1 + Y2)

        attention = tf.nn.sigmoid(Y_1 + Y_2)     # [B,1,C]
        return attention

# ------------------ Hybrid Spatial Attention (optimized) ------------------

class HybridSpatialAttention(layers.Layer):
    def __init__(self, d_model, units_q: int = 64,
                 fractal_dim=1.8, max_tokens: int = 256, geometry="auto", lam=0.0, **kw):
        """
        max_tokens: upper bound on H*W before tokenization. If exceeded, we downsample
                    (avg pool, stride 2) once or more to keep compute small.
        """
        super().__init__(**kw)
        #self.hybrid = HybridCAPE_ROAM_TF(d_model=d_model, lam=0.0)  # ROAM off by default
        
        self.hybrid = HybridCAPE_ROAM_TF(d_model=d_model,
                                         lam=lam,
                                         topk=8,
                                         geometry=geometry)
        
        self.atlas  = AtlasCodes(Nz=64, d_model=d_model)
        self.max_tokens = int(max_tokens)

        # quantum parameters ψ ∈ ℂ^{D×units_q}
        self.psi_real = self.add_weight(name="psi_real",
                                        shape=(d_model, units_q),
                                        initializer="glorot_uniform")
        self.psi_imag = self.add_weight(name="psi_imag",
                                        shape=(d_model, units_q),
                                        initializer="glorot_uniform")

        # fractal dimension scaling 1 ≤ S ≤ 2
        self.S = self.add_weight(name="fractal_scaling",
                                 shape=(units_q,),
                                 initializer=tf.keras.initializers.Constant(fractal_dim),
                                 constraint=MinMaxNorm(1.0, 2.0))

        # simple cross/output gates (channel–wise)
        self.cross_gate  = self.add_weight(name="cross_gate",
                                           shape=(d_model,),
                                           initializer="ones")
        self.output_gate = self.add_weight(name="output_gate",
                                           shape=(d_model,),
                                           initializer="ones")
        self.z_proj = LowRankDense(d_model, rank=max(1, d_model//8), use_bias=False)

    def _expand(self, v, target):
        """Broadcast [B,C] → [B,H,W,C] to match feature map."""
        if len(target.shape) == 4:
            v = v[:, None, None, :]                # [B,1,1,C]
            # Use target's *dynamic* H,W to avoid retracing
            H = tf.shape(target)[1]; W = tf.shape(target)[2]
            return tf.tile(v, [1, H, W, 1])
        return v                                   # already flat

    
    def _maybe_downsample(self, F):
        Hs, Ws = F.shape[1], F.shape[2]   # static dims if available
        if (Hs is not None) and (Ws is not None):
            tokens = Hs * Ws
            if tokens > self.max_tokens:
                F = tf.nn.avg_pool2d(F, ksize=2, strides=2, padding='SAME')  # /2
                Hs //= 2; Ws //= 2
                if Hs * Ws > self.max_tokens:
                    F = tf.nn.avg_pool2d(F, ksize=2, strides=2, padding='SAME')  # /4
        # else: unknown spatial dims at build time -> keep as-is
        return F


    

    def call(self, F1, F2, training=False):
        # simple spatial weighting
        def wm(F):
            avg = tf.reduce_mean(F, [1,2], keepdims=True)
            mx  = tf.reduce_max(F, [1,2], keepdims=True)
            mn  = tf.reduce_min(F, [1,2], keepdims=True)
            sm  = tf.reduce_sum(F, [1,2], keepdims=True)
            return tf.nn.softmax(avg+mx+mn+sm, -1) * F

        F1w = wm(F1)
        F2w = wm(F2)

        # ---- NEW: downsample before tokenization if too many tokens ----
        F1s = self._maybe_downsample(F1w)
        F2s = self._maybe_downsample(F2w)

        B, Hs, Ws, C = tf.unstack(tf.shape(F1s))
        B2, Hs2, Ws2, C2 = tf.unstack(tf.shape(F2s))

        XA = tf.reshape(F1s, [B, Hs*Ws, C])
        XB = tf.reshape(F2s, [B2, Hs2*Ws2, C2])
        #Z, _ = self.atlas(F1s)                     # atlas on the same (possibly downsampled) grid

        Z_static, _ = self.atlas(F1s)  # [B, Nz, C]
        summ = tf.concat([tf.reduce_mean(F1s, [1,2]), tf.reduce_mean(F2s, [1,2])], axis=-1)  # [B, 2C]
        z_bias = self.z_proj(summ)  # [B, C]
        Z = Z_static + z_bias[:, None, :]  # broadcast over Nz
        attn_core = self.hybrid(XA, XB, Z, training=training, return_certificates=False)

        #attn_core = self.hybrid(XA, XB, Z, training=training, return_certificates=False)  # [B,1,C]
        attn_core = attn_core[:, None, :, :]       # [B,1,1,C]

        # -------------- spectral/fractal channel attention --------------
        x1_flat = tf.signal.dct(tf.reduce_mean(F1, [1,2]), type=2, norm='ortho')  # [B,C]
        x2_flat = tf.signal.dct(tf.reduce_mean(F2, [1,2]), type=2, norm='ortho')

        psi_r = self.psi_real; psi_i = self.psi_imag
        ψ1 = tf.complex(tf.matmul(x1_flat, psi_r), tf.matmul(x1_flat, psi_i))
        ψ2 = tf.complex(tf.matmul(x2_flat, psi_r), tf.matmul(x2_flat, psi_i))

        p1 = tf.math.real(ψ1 * tf.math.conj(ψ1))             # |ψ|²
        p2 = tf.math.real(ψ2 * tf.math.conj(ψ2))

        scaled1 = tf.pow(p1 + 1e-8, self.S)                  # [B,units_q]
        scaled2 = tf.pow(p2 + 1e-8, self.S)

        att_q1 = tf.nn.softmax(scaled1, axis=-1)
        att_q2 = tf.nn.softmax(scaled2, axis=-1)

        v1 = tf.matmul(att_q1, tf.transpose(psi_r))          # [B,C]
        v2 = tf.matmul(att_q2, tf.transpose(psi_r))          # [B,C]

        v1 = self._expand(v1, F1)                            # [B,H,W,C]
        v2 = self._expand(v2, F2)

        # cross fusion
        cg   = tf.reshape(self.cross_gate,  (1,1,1,-1))
        gout = tf.reshape(self.output_gate, (1,1,1,-1))

        out1 = gout * (F2 * cg * v1)     # F2 guiding F1
        out2 = gout * (F1 * cg * v2)     # F1 guiding F2

        # fuse with CAPE/ROAM channel mask and return
        return out1 * attn_core + F1, out2 * attn_core + F2


In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K   # for get_uid()

def EMRC(x1, x2, filters, strides, base="emrc"):
    
    # Create a unique numeric id *per EMRC invocation*
    uid = K.get_uid(base)          # 0, 1, 2, … even across the whole model
    prefix = f"{base}{uid}_f{filters}"

    # Two symmetric RGSA calls with distinct names
    x1 = RGSA(
        x1,
        filters=filters,
        strides=strides,
        name=f"{prefix}_x1",
        use_projection=True,
    )
    x2 = RGSA(
        x2,
        filters=filters,
        strides=strides,
        name=f"{prefix}_x2",
        use_projection=True,
    )

    return x1, x2


In [ ]:
def info_fusion(x_1, x_2, filter):
    x11 = multi_kernel_groupwise_conv1(x_1, filters=filter, strides=(2,2))
    x21 = multi_kernel_groupwise_conv1(x_2, filters=filter, strides=(2,2))

    return x11, x21

def info_fusion1(x_1, x_2, filter):
    x11 = multi_kernel_groupwise_conv1(x_1, filters=filter, strides=(1,1))
    x21 = multi_kernel_groupwise_conv1(x_2, filters=filter,  strides=(1,1))

    return x11, x21

In [ ]:
def SIR(x1, x2, f):
    if f == 64:
        x111, x211 = info_fusion(x1, x2, 2 * f)
        x111, x211 = info_fusion(x111, x211, 4 * f)
        x111, x211 = info_fusion(x111, x211, 8 * f)
        return x111, x211

    elif f == 128:
        x111, x211 = info_fusion(x1, x2, 2 * f)
        x111, x211 = info_fusion(x111, x211, 4 * f)
        return x111, x211

    elif f == 256:
        x111, x211 = info_fusion(x1, x2, 2 * f)
        return x111, x211

    elif f == 512:
        x111, x211 = info_fusion1(x1, x2, f)
        return x111, x211

    else:
        raise ValueError(f"Unsupported fusion factor: {f}")

In [ ]:
def PCMFA(x1, x2, f):
    
    #x1, x2 = MACFusion(units=f)([x1, x2]) # First PCMFA
    #x1, x2 = MACFusion(units=f)([x1, x2]) # Second PCMFA

    attention = HybridSpatialAttention(d_model=f)
    x1, x2 = attention(x1, x2, training=True)
    x1, x2 = attention(x1, x2, training=True)
    
    #x1, x2 = MACFusion(units=f)([x1, x2]) # First PCMFA
    #x1, x2 = MACFusion(units=f)([x1, x2]) # Second PCMFA

    return x1, x2

In [ ]:
def EHF1(x1, x2, f, s):
    x1, x2 = EMRC(x1, x2, f, s) # Call EMRC module
    x1, x2 = PCMFA(x1, x2, f) # # Call PCMFA module
    return x1, x2    

In [ ]:
'''from tensorflow.keras.initializers import Ones, Constant

class LF(layers.Layer):
    def build(self, input_shape):
        # input_shape: list of two shapes, each (B, H, W, C)
        # we use a single scalar per tensor (broadcastable to all channels)
        

        self.global_scale1 = self.add_weight(
            name="global_scale1",
            shape=(1,1,1,1),
            initializer=Ones(),      # starts at 1.0
            trainable=True
        )
        # or, if you want a tiny deviation around 1:
        self.global_scale2 = self.add_weight(
            name="global_scale2",
            shape=(1,1,1,1),
            initializer=Constant(value=1.0),  
            # this gives values ~ N(1,0.01)
            trainable=True
        )

    def call(self, inputs):
        x1, x2 = inputs
        # scale each branch by its learnable scalar
        x1_scaled = x1 * self.global_scale1
        x2_scaled = x2 * self.global_scale2
        # concatenate along channel axis
        return layers.Concatenate(axis=-1)([x1_scaled, x2_scaled])'''

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class ModalityMixer(layers.Layer):
    def __init__(self, hidden_mul=2, p_min=0.0, p_max=0.5, warm_up=10000, **kw):
        super().__init__(**kw)
        self.hidden_mul = hidden_mul
        self.p_min = p_min
        self.p_max = p_max
        self.warm_up = warm_up
        self.step = tf.Variable(0, trainable=False, dtype=tf.int64)
        self.mlp = None
        self.reduce = None

    def build(self, input_shapes):
        C = int(input_shapes[0][-1])  # assumes x_a and x_b have same C
        self.mlp = tf.keras.Sequential([
            layers.Conv2D(self.hidden_mul * C, 1, activation="relu", groups = 8, use_bias=True),
            layers.Conv2D(2 * C, 1, groups = 8, use_bias=True),
        ])
        self.reduce = layers.Conv2D(C, 1, groups = 8,  padding="same", use_bias=False)

    def _drop_prob(self):
        progress = tf.cast(self.step, tf.float32) / tf.maximum(1.0, tf.cast(self.warm_up, tf.float32))
        return self.p_min + (self.p_max - self.p_min) * tf.clip_by_value(progress, 0.0, 1.0)

    def call(self, inputs, training=None):
        x_a, x_b = inputs
        if training:
            self.step.assign_add(1)

        eps = tf.cast(1e-6, x_a.dtype)

        # ---------- Boolean presence masks ----------
        pres_a = tf.reduce_any(tf.abs(x_a) > eps, axis=[1, 2, 3], keepdims=True)  # [B,1,1,1], bool
        pres_b = tf.reduce_any(tf.abs(x_b) > eps, axis=[1, 2, 3], keepdims=True)  # [B,1,1,1], bool

        # ---------- Per-example curriculum modality drop (only when both present) ----------
        if training:
            B = tf.shape(x_a)[0]
            p_drop = self._drop_prob()
            u = tf.random.uniform([B], 0.0, 1.0)          # [B]
            want_drop = u < p_drop                         # [B], bool
            both = tf.squeeze(tf.logical_and(pres_a, pres_b), [1, 2, 3])  # [B], bool
            do = tf.logical_and(want_drop, both)           # [B], bool

            coin = tf.random.uniform([B], 0.0, 1.0)
            drop_a = tf.reshape(tf.logical_and(do, coin < 0.5), [B, 1, 1, 1])
            drop_b = tf.reshape(tf.logical_and(do, coin >= 0.5), [B, 1, 1, 1])

            x_a = tf.where(drop_a, tf.zeros_like(x_a), x_a)
            x_b = tf.where(drop_b, tf.zeros_like(x_b), x_b)
            pres_a = tf.where(drop_a, tf.zeros_like(pres_a), pres_a)
            pres_b = tf.where(drop_b, tf.zeros_like(pres_b), pres_b)

        # ---------- Reliability features ----------
        def rms(x):
            return tf.sqrt(tf.reduce_mean(tf.square(x), axis=[1, 2], keepdims=True) + eps)

        def spatial_var(x):
            m = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
            return tf.reduce_mean(tf.square(x - m), axis=[1, 2], keepdims=True) + eps

        feats = tf.concat([rms(x_a), rms(x_b), spatial_var(x_a), spatial_var(x_b)], axis=-1)  # [B,1,1,4C]

        logits = self.mlp(feats)  # [B,1,1,2C]
        C = tf.shape(x_a)[-1]
        logits = tf.reshape(logits, tf.concat([tf.shape(logits)[:-1], [2, C]], axis=0))  # [B,1,1,2,C]

        # ---------- Mask absent modality with very negative ----------
        very_neg = tf.cast(-1e4, logits.dtype)  # safe for fp16/bf16/fp32
        mask = tf.stack([tf.cast(pres_a, logits.dtype), tf.cast(pres_b, logits.dtype)], axis=-2)  # [B,1,1,2,1]
        mask = tf.tile(mask, [1, 1, 1, 1, C])                                                   # [B,1,1,2,C]
        masked_logits = tf.where(mask > 0.5, logits, very_neg)

        weights = tf.nn.softmax(masked_logits, axis=-2)  # [B,1,1,2,C]
        w_a = weights[..., 0, :]
        w_b = weights[..., 1, :]

        # ---------- Fuse & reduce ----------
        x_sum = w_a * x_a + w_b * x_b
        mix = tf.concat([x_sum, x_a, x_b], axis=-1)  # 3C
        return self.reduce(mix)


In [ ]:
def EHF2(x1, x2, f, s):
    x1, x2 = EMRC(x1, x2, f, s) # Call EMRC module
    
    x1 = layers.BatchNormalization()(x1)
    x2 = layers.BatchNormalization()(x2)
    
    x1, x2 = PCMFA(x1, x2, f) # Call PCMFA module
    
    x111, x211 = SIR(x1, x2, f) ## Call SIR module

    #x1 = layers.Concatenate(axis = -1)([x1, x2])
    
    
    #x1 = LF()([x1, x2])  # Use Learnable Late Fusion (LF) for capturing multimodal information

    x1 = ModalityMixer(hidden_mul=2, p_min=0.05, p_max=0.5, warm_up=2000)([x1, x2])
    
    '''x1 = layers.Conv2D(f, kernel_size=(1, 1), groups=8, 
                       padding="same")(x1) '''
    
    return x1, x111, x211    

In [ ]:
def EHF(x1, x2, filters, strides1, strides2):
    if filters == 64:
        x1, x2 = EHF1(x1, x2, filters, strides2)
        x1, x2 = EHF1(x1, x2, filters, strides2)
        return x1, x2

    elif filters == 128:
        x1, x2 = EHF1(x1, x2, filters, strides1)
        x1, x111, x211 = EHF2(x1, x2, filters, strides2)
        return x1, x111, x211

    elif filters == 256:
        x1, x2 = EHF1(x1, x2, filters, strides1)
        x1, x111, x211 = EHF2(x1, x2, filters, strides2)
        return x1, x111, x211

    elif filters == 512:
        x1, x2 = EHF1(x1, x2, filters, strides1)
        x1, x111, x211 = EHF2(x1, x2, filters, strides2)
        return x1, x111, x211

    else:
        return x1, x2


In [ ]:
def residual_GLC_branch1(inputs1, inputs2, inputs3, inputs4):

    # Modality input 1
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs1)
    x1 = BatchNormalization()(x1)
    x1 = tf.keras.layers.Activation('relu')(x1)
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)

    # Modality input 2
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs2)
    x2 = BatchNormalization()(x2)
    x2 = tf.keras.layers.Activation('relu')(x2)
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)

    
    x1, x2 = EHF(x1, x2, filters=64, strides1=1, strides2=1)

    x1, x111, x211 = EHF(x1, x2, filters = 128, strides1=1, strides2=2)


    # Modality input 3
    x3 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs3)
    x3 = BatchNormalization()(x3)
    x3 = tf.keras.layers.Activation('relu')(x3)
    x3 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x3)

    x2 = multi_kernel_groupwise_conv3(x3, filters=256,  strides=2)
    
    x1, x1111, x2111 = EHF(x1, x2, filters = 256, strides1=1, strides2=2)

    
    # Modality input 4
    x4 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs4)
    x4 = BatchNormalization()(x4)
    x4 = tf.keras.layers.Activation('relu')(x4)
    x4 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x4)
    
    
    x2 = multi_kernel_groupwise_conv3(x4, filters=512, strides=2)
    x2 = multi_kernel_groupwise_conv3(x2, filters=512, strides=2)
    
    #x2 = RGSA(x2, filters=512, strides=(2, 2), use_projection=True)

    x1, x11111, x21111 = EHF(x1, x2, filters = 512, strides1=1, strides2=2)
    
    
    return x1, x111, x211, x1111, x2111, x11111, x21111

In [ ]:
'''class CIndexCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Predict risk scores on the full validation set
        pred = self.model.predict(X_val_multi, verbose=0).reshape(-1)
        #risk_scores = preds[2].reshape(-1)
        durs = y_val_tab[:, 0]
        evts = y_val_tab[:, 1].astype(bool)
        ci = concordance_index(durs, -pred, evts)
        logs = logs or {}
        logs['val_c_index'] = ci
        print(f" — val_c_index: {ci:.4f}")
'''

'''class CIndexCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_inputs, val_cox_y):
        """
        val_inputs: list of your four validation inputs,
                    e.g. [X_val_s, X_val_h1, X_val_multi, X_val_m1]
        val_cox_y:  the (n,2) array of [durations, events] for the Cox head
        """
        super().__init__()
        self.val_inputs = val_inputs
        self.val_cox_y   = val_cox_y

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # 1) predict returns a list of outputs, one per head
        preds = self.model.predict(self.val_inputs, batch_size = 4, verbose=0)
        # 2) pick out the Cox‐head predictions (assuming it's the 3rd head: index 2)
        risk_scores = preds[2].reshape(-1)

        # 3) pull durations/events
        durations = self.val_cox_y[:, 0]
        events    = self.val_cox_y[:, 1].astype(bool)

        # 4) compute C‐index
        ci = concordance_index(durations, -risk_scores, events)
        logs['val_c_index'] = ci

        print(f" — val_c_index: {ci:.4f}")


#cidx_cb = CIndexCallback()

val_inputs = [X_val_s, X_val_h1, X_val_multi, X_val_m1]
# the Cox‐head targets for val:
val_cox_y = y_val_multi  # shape (n_val, 2)

cidx_cb = CIndexCallback(val_inputs, val_cox_y)
'''

In [ ]:
'''def neg_log_partial_likelihood(y_true, y_pred):
    eps = 1e-8

    # unpack
    durations = y_true[:, 0]
    events    = y_true[:, 1]

    # sort by descending time
    order = tf.argsort(durations, direction="DESCENDING")
    p = tf.gather(tf.reshape(y_pred, [-1]), order)

    # clip the raw predictions to a safe window before exp
    p = tf.clip_by_value(p, -10.0, 10.0)

    # hazard ratio
    hr = tf.exp(p)
    hr = tf.clip_by_value(hr, eps, 1e6)

    # cumulative hazard
    cum_h = tf.cumsum(hr) + eps
    log_cum = tf.math.log(cum_h)

    diff = p - log_cum
    loss = -tf.reduce_sum(diff * tf.gather(events, order))

    return loss / (tf.reduce_sum(events) + eps)
'''

In [ ]:
#def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
def build_model():

    input_shape=(128, 128, 3)
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    inputs3 = Input(shape=input_shape)
    inputs4 = Input(shape=input_shape)
    
    import tensorflow.keras.layers as L
    
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    
    #x1, x2 = residual_GLC_branch1(inputs1, inputs2)
    
    x1, x111, x211, x1111, x2111, x11111, x21111 = residual_GLC_branch1(inputs1, inputs2, inputs3, inputs4)
    #print('x:',x.shape)
    
    con1 = tf.keras.layers.Concatenate(axis=-1)([x111, x1111, x11111])
    con2 = tf.keras.layers.Concatenate(axis=-1)([x211, x2111, x21111])

    con = tf.keras.layers.Concatenate(axis=-1)([x1, con1, con2])

    #con = tf.keras.layers.Concatenate(axis=-1)([x1, x111, x1111, x11111, x211, x2111, x21111])
    
    con = tf.keras.layers.Dropout(0.25)(con, training = True)  ## MCD ####
    
    x = GlobalAveragePooling2D()(con)
    
    
    outputs1 = Dense(5, activation='softmax')(x) # Output layer for modality input 1
    outputs2 = Dense(7, activation='softmax')(x) # Output layer for modality input 2
    outputs3 = Dense(2, activation="sigmoid")(x) # Output layer for modality input 3
    outputs4 = Dense(2, activation='sigmoid')(x) # Output layer for modality input 4
    
    # Create the model
    model = Model([inputs1, inputs2, inputs3, inputs4], [outputs1, outputs2, outputs3, outputs4])
    #return model
    #print(model.summary())

    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.optimizers.schedules import ExponentialDecay

    initial_gamma1 = 0.25
    initial_gamma2 = 0.25
    initial_gamma3 = 0.25
    
    
    
    from tensorflow.keras.optimizers import Adam

    #opt = Adam(learning_rate=1e-4, clipnorm=1.0)
    
    #opt = Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.9, epsilon=None, amsgrad=False)
    
    # Compile the model with the custom optimizer
    model.compile(optimizer='adam', loss=['categorical_crossentropy', 'categorical_crossentropy', #neg_log_partial_likelihood, 
                                          'binary_crossentropy', 'binary_crossentropy'],
                  loss_weights=[initial_gamma1, initial_gamma2, initial_gamma3, (1 -  (initial_gamma1 + initial_gamma2 + initial_gamma3))],
                  metrics=['accuracy','accuracy','accuracy','accuracy']
                  #metrics={'h':    ['accuracy', tf.keras.metrics.AUC(name='auc')], 's':    ['accuracy', tf.keras.metrics.AUC(name='auc')],
            #'multi':['accuracy', tf.keras.metrics.AUC(name='auc')], 'm1':   ['accuracy', tf.keras.metrics.AUC(name='auc')]}
                 )
       
    return model

In [ ]:
model = build_model()

print(model.summary())

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best1_model.keras'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           #frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                            mode='min',
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=60, verbose=1)
    return es_callback



from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                              patience=50, verbose = 1, min_lr=0.000001)

checkpoint_callback = checkpoint_callback()

early_stopping = early_stopping(patience=100)
callbacks = [#cidx_cb, 
             checkpoint_callback, early_stopping, reduce_lr]



In [ ]:
# Fit the model with callbacks
history = model.fit([X_train_s, X_train_h, X_train_multi, X_train_m1], [y_train_s, y_train_h, y_train_multi, y_train_m1],
                    epochs=200,
                    validation_data=([X_val_s, X_val_h1, X_val_multi, X_val_m1], [y_val_s, y_val_h1, y_val_multi, y_val_m1]), verbose=1, 
                    shuffle=True, batch_size=16,
                    callbacks=callbacks) 

In [ ]:
model.evaluate([X_test_s, X_test_h1, X_test_multi, X_test_m1], [y_test_s, y_test_h1, y_test_multi, y_test_m1])

In [ ]:
import numpy as np
import tensorflow as tf
import random

# 1) Generate or define your five seeds
import random

# 1) Generate five random 32-bit integer seeds
#    (you can set a seed here if you want reproducible seed-generation)
seeds = np.random.randint(0, 2**31 - 1, size=5).tolist()
print("Using seeds:", seeds)


# 2) Prepare empty lists for each modality
modalities = ['Dermoscopy', 'Cytology', 'ECG', 'EHR']
stats = {
    mod: {
        'accs': [],
        'aucs': []
    } for mod in modalities
}

for seed in seeds:
    # ── reproducibility ───────────────────────────────────────────
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # ── build & compile ───────────────────────────────────────────
    model = build_model()  # your model builder
    model.compile(
        optimizer='adam',
        loss={
            'h':    'binary_crossentropy',
            's':    'binary_crossentropy',
            'multi':'binary_crossentropy',
            'm1':   'binary_crossentropy'
        },
        metrics={
            'h':    ['accuracy', tf.keras.metrics.AUC(name='auc')],
            's':    ['accuracy', tf.keras.metrics.AUC(name='auc')],
            'multi':['accuracy', tf.keras.metrics.AUC(name='auc')],
            'm1':   ['accuracy', tf.keras.metrics.AUC(name='auc')]
        }
    )

    # ── fit ───────────────────────────────────────────────────────
    model.fit(
        [X_train_h, X_train_s, X_train_multi, X_train_m1],
        [y_train_h, y_train_s, y_train_multi, y_train_m1],
        epochs=200,
        validation_data=(
            [X_val_h1, X_val_s, X_val_multi, X_val_m1],
            [y_val_h1, y_val_s, y_val_multi, y_val_m1]
        ),
        verbose=0,
        shuffle=True,
        callbacks=callbacks
    )

    # ── evaluate ─────────────────────────────────────────────────
    names, results = model.metrics_names, model.evaluate(
        [X_test_h1, X_test_s, X_test_multi, X_test_m1],
        [y_test_h1, y_test_s, y_test_multi, y_test_m1],
        verbose=0
    )
    res = dict(zip(names, results))

    # ── collect per‐modality ──────────────────────────────────────
    for mod in modalities:
        stats[mod]['accs'].append(res[f"{mod}_accuracy"])
        stats[mod]['aucs'].append(res[f"{mod}_auc"])

# 3) Compute & print modality-wise mean ± std
for mod in modalities:
    accs = np.array(stats[mod]['accs'])
    aucs = np.array(stats[mod]['aucs'])
    print(f"Modality {mod.upper()}:")
    print(f"  ACC: {accs.mean():.4f} ± {accs.std():.4f}")
    print(f"  AUC: {aucs.mean():.4f} ± {aucs.std():.4f}\n")
